# S3.11 — String Functions
**Date completed:** October 2026  
**Status:** Completed  
**Key concept:** Silver layer string cleaning patterns

In [0]:
# ============================================================
# S3.11 — Cell 2
# String Functions: Silver-Layer Data Cleaning
#
# Goal:
# 1. Use upper(), lower(), and initcap() for case handling.
# 2. Use trim(), ltrim(), and rtrim() for whitespace cleaning.
# 3. Use split() to break structured strings into components.
# 4. Use concat() / concat_ws() to combine string values.
# 5. Use regexp_replace() to clean inconsistent text.
# 6. Safely access split-array elements using get().
# 7. Combine string functions into a Silver-style
#    transformation.
# 8. Apply the concepts to samples.tpch.customer.
#
# Environment:
# Databricks Serverless
#
# Important:
# String standardisation must follow business rules.
# Do not automatically change casing, remove characters,
# or rewrite source values without understanding their meaning.
# ============================================================

import pyspark.sql.functions as F

print("=== S3.11: STRING FUNCTIONS ===")
print()

In [0]:
# ------------------------------------------------------------
# Step 1 — Create Controlled Messy Bronze-Like Data
#
# We intentionally create inconsistent strings so that each
# cleaning function can be observed clearly.
#
# customer_name format:
# LAST_NAME, FIRST_NAME
# ------------------------------------------------------------

messy_data = [

    (1, "  SMITH,  JOHN  ", "  DELHI-NCR  ", "9876-543-210", "INV-2026-001"),

    (2, "  jones,   mary  ", "  mumbai  ", "98765 43210", "INV-2026-002"),

    (3, "  KUMAR, raj  ", "  PUNE-CITY  ", "(987)6543210", "INV-2026-003"),

    (4, "  Chen,  Li  ", "  chennai  ", "987.654.3210", "INV-2026-004"),

    (5, None, None, None, None)]


df = spark.createDataFrame(
    messy_data,
    ["id", "customer_name", "city", "phone", "invoice_id"])

print("=== STEP 1: RAW BRONZE-LIKE DATA ===")

df.show(truncate=False)

source_count = df.count()

print(f"\nSource row count: {source_count}")

print()


In [0]:
# ------------------------------------------------------------
# Step 2 — Case Functions
#
# upper()
#     → Converts text to uppercase.
#
# lower()
#     → Converts text to lowercase.
#
# initcap()
#     → Capitalises words.
#
# We combine trim() here so surrounding spaces do not make
# the output harder to interpret.
# ------------------------------------------------------------

print("=== STEP 2: CASE FUNCTIONS ===")


df_case = df.select(

    F.col("id"),

    F.col("city").alias("original_city"),

    F.upper(F.trim(F.col("city"))).alias("city_upper"),

    F.lower(F.trim(F.col("city"))).alias("city_lower"),

    F.initcap(F.trim(F.col("city"))).alias("city_initcap"))


df_case.show(truncate=False)

print("""
Important:

upper()
→ UPPERCASE

lower()
→ lowercase

initcap()
→ Capitalises words according to Spark's string rules.

Case conversion should be used only when the target
business standard requires it.
""")


In [0]:
# ------------------------------------------------------------
# Step 3 — Trim Functions
#
# trim()
#     → Removes leading and trailing spaces.
#
# ltrim()
#     → Removes leading spaces.
#
# rtrim()
#     → Removes trailing spaces.
#
# These functions do NOT automatically remove unnecessary
# spaces inside the string.
# ------------------------------------------------------------

print("=== STEP 3: TRIM FUNCTIONS ===")


df_trim = df.select(

    F.col("id"),

    F.col("customer_name").alias("original"),

    F.trim(F.col("customer_name")).alias("trimmed"),

    F.ltrim(F.col("customer_name")).alias("left_trimmed"),

    F.rtrim(F.col("customer_name")).alias("right_trimmed"))


df_trim.show(truncate=False)

print("""
Important:

trim()
→ removes spaces from BOTH ends.

ltrim()
→ removes spaces from the LEFT.

rtrim()
→ removes spaces from the RIGHT.

They do not automatically normalise internal spaces.
""")


In [0]:
# ------------------------------------------------------------
# Step 4 — regexp_replace(): Clean / Standardise Strings
#
# regexp_replace() replaces text matching a regular-expression
# pattern.
#
# Real example:
# Phone numbers arrive using:
#
# -
# spaces
# ()
# .
#
# The target requirement is digits only.
# ------------------------------------------------------------

print("=== STEP 4: regexp_replace() ===")


df_phone = df.select(

    F.col("id"),

    F.col("phone").alias("raw_phone"),

    F.regexp_replace(F.col("phone"),r"[^0-9]","").alias("clean_phone"))

df_phone.show(truncate=False)

print("""
Regex:

[^0-9]

means:

Replace every character that is NOT a digit.

Example:

(987)6543210
      ↓
9876543210
""")


In [0]:
# ------------------------------------------------------------
# Step 5 — Normalise Internal Whitespace
#
# trim() only removes spaces from the ends.
#
# regexp_replace(..., r"\s+", " ")
#
# converts multiple whitespace characters inside the string
# into a single space.
# ------------------------------------------------------------

print("=== STEP 5: NORMALISE INTERNAL WHITESPACE ===")

df_whitespace = df.select(
    "id",

    F.col("customer_name").alias("original_name"),

    F.trim(F.regexp_replace(F.col("customer_name"),r"\s+"," ")).alias("normalised_name"))

df_whitespace.show(truncate=False)

print()


In [0]:
# ------------------------------------------------------------
# Step 6 — split(): Break a String into Parts
#
# Source customer_name format:
#
# LAST_NAME, FIRST_NAME
#
# Example:
#
# SMITH, JOHN
#
# split() returns an ARRAY.
#
# Because array positions may be missing in malformed data,
# we use F.get() for safe zero-based access.
#
# This is especially important in ANSI-enabled environments.
# ------------------------------------------------------------

print("=== STEP 6: split() ===")

# First normalise repeated whitespace.

normalised_name = F.trim(F.regexp_replace(F.col("customer_name"),r"\s+"," "))

# Split around the comma.
#
# \s*,\s*
#
# means:
# optional whitespace + comma + optional whitespace

name_parts = F.split(normalised_name,r"\s*,\s*")

# Safe zero-based array access.

last_name_expr = F.initcap(F.get(name_parts,0))

first_name_expr = F.initcap(F.get(name_parts,1))

df_split = df.select(
    F.col("id"),

    normalised_name.alias("full_name_source"),

    last_name_expr.alias("last_name"),

    first_name_expr.alias("first_name"))


df_split.show(truncate=False)

print("""
Source format:

LAST_NAME, FIRST_NAME

Example:

SMITH, JOHN
     ↓
split()
     ↓
["SMITH", "JOHN"]

F.get(..., 0)
→ SMITH

F.get(..., 1)
→ JOHN

F.get() is safer than direct array indexing when an
expected element may be missing.
""")

In [0]:
# ------------------------------------------------------------
# Step 7 — concat() and concat_ws()
#
# concat()
#     → Concatenates expressions directly.
#
# concat_ws()
#     → Concatenates using a separator.
#
# We want the target name format:
#
# FIRST_NAME LAST_NAME
#
# rather than the Bronze format:
#
# LAST_NAME, FIRST_NAME
# ------------------------------------------------------------

print("=== STEP 7: concat() AND concat_ws() ===")


df_concat = df.select(

    "id",

    first_name_expr.alias("first_name"),

    last_name_expr.alias("last_name"),

    # concat() requires us to provide the separator
    # explicitly using lit(" ").

    F.when(
        F.col("customer_name").isNull(),
        F.lit(None).cast("string")
    )
    .otherwise(
        F.concat(
            first_name_expr,
            F.lit("-"),
            last_name_expr
        )).alias("name_using_concat"),

    # concat_ws() provides the separator directly.

    F.when(
        F.col("customer_name").isNull(),
        F.lit(None).cast("string")
    )
    .otherwise(
        F.concat_ws(
            " ",
            first_name_expr,
            last_name_expr))
    .alias("name_using_concat_ws"))

df_concat.show(truncate=False)

print("""
concat():

concat(
    first_name,
    lit(" "),
    last_name
)

concat_ws():

concat_ws(
    " ",
    first_name,
    last_name
)

concat_ws() is often cleaner when joining several
string values using the same separator.
""")


In [0]:
# ------------------------------------------------------------
# Step 8 — split() Structured Invoice ID
#
# invoice_id format:
#
# INV-2026-001
#
# We can split the value into:
#
# invoice_type
# invoice_year
# invoice_sequence
#
# Again we use F.get() instead of unsafe direct indexing.
# ------------------------------------------------------------

print("=== STEP 8: SPLIT INVOICE ID ===")


invoice_parts = F.split(F.col("invoice_id"),"-")

df_invoice = df.select(

    "id",

    "invoice_id",

    F.get(invoice_parts,0).alias("invoice_type"),

    F.get(invoice_parts,1).alias("invoice_year"),

    F.get(invoice_parts,2).alias("invoice_sequence"))

df_invoice.show(truncate=False)

print()


In [0]:
# ------------------------------------------------------------
# Step 9 — Full Silver-Style String Cleaning
#
# Combine:
#
# trim()
# regexp_replace()
# split()
# get()
# initcap()
# concat_ws()
# upper()
#
# into one transformation.
# ------------------------------------------------------------

print("=== STEP 9: FULL SILVER-STYLE CLEANING ===")

# ------------------------------------------------------------
# Customer Name
#
# Bronze:
# "  SMITH,  JOHN  "
#
# Target:
# "John Smith"
#
# Preserve NULL when source name is NULL.
# ------------------------------------------------------------

customer_name_clean = (

    F.when(
        F.col("customer_name").isNull(),
        F.lit(None).cast("string")
    )

    .otherwise(
        F.concat_ws(
            " ",
            first_name_expr,
            last_name_expr
        )
    )
)


# ------------------------------------------------------------
# City
#
# Example:
#
# "  DELHI-NCR  "
#
# Step 1:
# Replace "-" with space.
#
# Step 2:
# Collapse repeated whitespace.
#
# Step 3:
# Trim.
#
# Step 4:
# initcap().
# ------------------------------------------------------------

city_clean = F.initcap(

    F.trim(
        F.regexp_replace(
            F.regexp_replace(
                F.col("city"),"-"," "),r"\s+"," ")))

# ------------------------------------------------------------
# Phone
#
# Keep digits only.
# ------------------------------------------------------------

phone_clean = F.regexp_replace(F.col("phone"),r"[^0-9]","")

# ------------------------------------------------------------
# Invoice ID
#
# Remove surrounding whitespace if it exists and standardise
# alphabetic characters to uppercase.
# ------------------------------------------------------------

invoice_clean = F.upper(F.trim(F.col("invoice_id")))

df_silver = df.select(

    F.col("id"),

    customer_name_clean.alias("customer_name"),
    
    city_clean.alias("city"),

    phone_clean.alias("phone"),

    invoice_clean.alias("invoice_id"))

print("\nCleaned Silver-style data:")

df_silver.show(truncate=False)

print()


In [0]:
# ------------------------------------------------------------
# Step 10 — Validate Cleaning
# ------------------------------------------------------------

print("=== STEP 10: VALIDATION ===")

silver_count = df_silver.count()

assert silver_count == source_count, ("Row count changed during string cleaning")

expected_columns = [
    "id",
    "customer_name",
    "city",
    "phone",
    "invoice_id"
]

assert df_silver.columns == expected_columns, ("Unexpected Silver column structure")

# Verify that non-NULL phone values contain digits only.

invalid_phone_count = (
    df_silver
    .filter(
        F.col("phone").isNotNull()
        &
        ~F.col("phone").rlike(r"^[0-9]+$")).count())

assert invalid_phone_count == 0, ("Non-numeric characters remain in cleaned phone numbers")

print("String-cleaning validation passed.")
print()


In [0]:
# ------------------------------------------------------------
# Step 11 — Apply String Functions to Real TPC-H Data
#
# We now apply several techniques to the actual
# samples.tpch.customer table.
#
# This is not because TPC-H is necessarily dirty.
#
# The purpose is to demonstrate the same transformation
# pattern against a real benchmark table.
# ------------------------------------------------------------

print("=== STEP 11: samples.tpch.customer PRACTICAL ===")

df_customer = spark.table("samples.tpch.customer")

print("\nSource customer schema:")

df_customer.printSchema()

df_customer_clean = df_customer.select(

    F.col("c_custkey").alias("customer_id"),

    F.trim(F.col("c_name")).alias("customer_name"),

    F.trim(F.col("c_address")).alias("address"),

    F.regexp_replace(F.col("c_phone"), r"[^0-9]","").alias("phone_digits"),

    F.upper(F.trim(F.col("c_mktsegment"))).alias("market_segment"))

print("\nTPC-H customer cleaned projection:")

df_customer_clean.show(5,truncate=False)

print()


In [0]:
# ------------------------------------------------------------
# Step 12 — Important Business Considerations
# ------------------------------------------------------------

print("=== STEP 12: BUSINESS CONSIDERATIONS ===")

print("""
String cleaning is NOT simply:

"make everything uppercase"
or
"remove every special character"

Cleaning must preserve business meaning.

Examples:

Names:
- initcap() may be useful for presentation.
- But real names can have legitimate casing that automatic
  title-casing may not preserve perfectly.

Phone numbers:
- Removing formatting may be appropriate for a canonical
  numeric representation.
- Country codes and leading zeros must still be understood.

Addresses:
- Do not remove punctuation blindly.
- Punctuation may contain meaningful address information.

IDs:
- Do not remove characters unless the ID contract permits it.

NULL:
- Most Spark string functions propagate NULL.
- Do not automatically convert NULL into an empty string.
""")


In [0]:
# ------------------------------------------------------------
# Step 13 — Transformations vs Actions
# ------------------------------------------------------------

print("=== STEP 13: TRANSFORMATIONS VS ACTIONS ===")

print("""
Column-expression functions used:

upper()
lower()
initcap()
trim()
ltrim()
rtrim()
regexp_replace()
split()
get()
concat()
concat_ws()

These functions create Spark Column expressions.

They are used inside transformations such as:

select()
withColumn()

Transformations used:

select()

Actions used:

show()
count()

Spark still follows lazy evaluation.
""")

In [0]:
# ------------------------------------------------------------
# Step 14 — Final Learning Summary
# ------------------------------------------------------------

print("=== FINAL LEARNING SUMMARY ===")


print("""
upper():
- Converts text to uppercase.

lower():
- Converts text to lowercase.

initcap():
- Capitalises words.

trim():
- Removes leading and trailing whitespace.

ltrim():
- Removes leading whitespace.

rtrim():
- Removes trailing whitespace.

regexp_replace():
- Replaces text matching a regular-expression pattern.
- Useful for string standardisation and cleaning.

split():
- Converts a string into an array based on a delimiter/pattern.

get():
- Safely retrieves an array element using a zero-based index.
- Useful when the expected array position may not exist.

concat():
- Joins multiple string expressions.

concat_ws():
- Joins multiple string expressions using a separator.

Important:

String functions normally create Column expressions.
They do not independently trigger Spark execution.


Silver-layer principle:

Raw String
    ↓
Understand business format
    ↓
Trim / normalise
    ↓
Parse structured components
    ↓
Standardise case / characters when required
    ↓
Validate
    ↓
Silver-ready value
""")


print("=== S3.11 CELL 2 COMPLETED ===")

# S3.11 — Key Takeaways: String Functions

## 1. Case Functions

| Function | Example | Result |
|---|---|---|
| `upper()` | `upper("delhi")` | `"DELHI"` |
| `lower()` | `lower("DELHI")` | `"delhi"` |
| `initcap()` | `initcap("delhi ncr")` | `"Delhi Ncr"` |

### Purpose

```text
upper()
→ Standardise text to uppercase

lower()
→ Standardise text to lowercase

initcap()
→ Capitalise words
```

Case standardisation should follow the business requirement.

For example, automatically applying `initcap()` to every person's name
may not preserve all real-world naming conventions correctly.

---

# 2. Trim Functions

| Function | Purpose |
|---|---|
| `trim()` | Remove leading and trailing whitespace |
| `ltrim()` | Remove leading whitespace |
| `rtrim()` | Remove trailing whitespace |

Example:

```python
F.trim(
    F.col("customer_name")
)
```

### Important

These functions remove whitespace from the edges.

They do not automatically clean repeated whitespace inside the string.

Example:

```text
"  JOHN    SMITH  "
        ↓
trim()
        ↓
"JOHN    SMITH"
```

The internal repeated spaces remain.

---

# 3. regexp_replace()

`regexp_replace()` replaces text matching a regular-expression pattern.

Syntax:

```python
F.regexp_replace(
    F.col("column_name"),
    pattern,
    replacement
)
```

### Example — Keep digits only

```python
F.regexp_replace(
    F.col("phone"),
    r"[^0-9]",
    ""
)
```

Example:

```text
(987)654-3210
      ↓
9876543210
```

---

# 4. Useful regexp_replace() Patterns

### Keep digits only

```python
F.regexp_replace(
    F.col("phone"),
    r"[^0-9]",
    ""
)
```

Meaning:

```text
Replace every character that is NOT 0–9.
```

---

### Normalise repeated whitespace

```python
F.regexp_replace(
    F.col("name"),
    r"\s+",
    " "
)
```

Example:

```text
"John     Smith"
       ↓
"John Smith"
```

This **normalises repeated whitespace**.

It does not remove every space.

---

### Remove non-alphanumeric characters

```python
F.regexp_replace(
    F.col("value"),
    r"[^a-zA-Z0-9]",
    ""
)
```

### Important

Do not apply this pattern blindly.

Special characters can have valid business meaning in:

```text
names
addresses
IDs
product codes
email addresses
```

Cleaning rules should follow the source-data contract.

---

# 5. split()

`split()` divides a string using a delimiter or regular-expression pattern.

It returns an ARRAY.

Example source:

```text
SMITH, JOHN
```

Example:

```python
name_parts = F.split(
    F.col("customer_name"),
    r"\s*,\s*"
)
```

Conceptually:

```text
"SMITH, JOHN"
      ↓
split()
      ↓
["SMITH", "JOHN"]
```

---

# 6. Safe Array Access with get()

Because `split()` returns an array, we need to retrieve its elements.

In our Databricks/ANSI environment, we used:

```python
F.get(
    name_parts,
    0
)
```

and:

```python
F.get(
    name_parts,
    1
)
```

`F.get()` uses a zero-based index.

Example:

```text
["SMITH", "JOHN"]

index 0
→ SMITH

index 1
→ JOHN
```

### Why we used F.get()

If the expected array position does not exist, `F.get()` safely returns
`NULL`.

This is preferable to assuming every incoming string contains the
expected number of elements.

---

# 7. concat()

`concat()` joins multiple string expressions directly.

Example:

```python
F.concat(
    F.col("first_name"),
    F.lit(" "),
    F.col("last_name")
)
```

Result:

```text
John + " " + Smith
        ↓
John Smith
```

When using `concat()`, separators such as spaces must be supplied explicitly.

---

# 8. concat_ws()

`concat_ws()` means:

```text
concat with separator
```

Example:

```python
F.concat_ws(
    " ",
    F.col("first_name"),
    F.col("last_name")
)
```

Result:

```text
John Smith
```

Another example:

```python
F.concat_ws(
    "-",
    F.col("city"),
    F.col("region")
)
```

could produce:

```text
Delhi-NCR
```

`concat_ws()` is convenient when multiple values should use the same separator.

---

# 9. Parsing a Structured String

We also used string functions to parse:

```text
INV-2026-001
```

Using:

```python
invoice_parts = F.split(
    F.col("invoice_id"),
    "-"
)
```

Then:

```python
F.get(invoice_parts, 0)
F.get(invoice_parts, 1)
F.get(invoice_parts, 2)
```

produces conceptually:

```text
INV-2026-001
     ↓
["INV", "2026", "001"]

invoice_type
→ INV

invoice_year
→ 2026

invoice_sequence
→ 001
```

This is useful when a source field contains multiple structured components.

---

# 10. Combining String Functions

String functions can be nested together.

Example:

```python
F.initcap(
    F.trim(
        F.regexp_replace(
            F.col("city"),
            "-",
            " "
        )
    )
)
```

Read from the inside out:

```text
Source city
     ↓
Replace "-"
     ↓
Trim surrounding whitespace
     ↓
Standardise case
     ↓
Clean city
```

---

# 11. Customer Name Cleaning Pattern

Our source format was:

```text
LAST_NAME, FIRST_NAME
```

Example:

```text
SMITH, JOHN
```

Simply removing the comma would produce:

```text
Smith John
```

but this would still be:

```text
LastName FirstName
```

If the target requirement is:

```text
John Smith
```

we need to understand the source structure first.

Correct conceptual pattern:

```text
"SMITH, JOHN"
      ↓
Normalise whitespace
      ↓
split(",")
      ↓
last_name = SMITH
first_name = JOHN
      ↓
Standardise case
      ↓
concat_ws(" ", first_name, last_name)
      ↓
"John Smith"
```

This demonstrates an important data-engineering principle:

**Parse according to business meaning before reconstructing the target value.**

---

# 12. NULL Behaviour

Many Spark string functions used in this lesson propagate NULL input.

Example:

```text
customer_name = NULL
        ↓
trim(customer_name)
        ↓
NULL
```

However, NULL behaviour should be understood for the particular
function being used rather than assuming every string function behaves
identically.

Do not automatically change:

```text
NULL
```

into:

```text
""
```

because NULL and an empty string represent different information.

---

# 13. String Functions Are Column Expressions

Functions such as:

```text
upper()
lower()
initcap()
trim()
ltrim()
rtrim()
regexp_replace()
split()
get()
concat()
concat_ws()
```

produce Spark Column expressions.

They can be used inside transformations such as:

```python
df.select(...)
```

or:

```python
df.withColumn(...)
```

They do not independently trigger Spark execution.

---

# 14. Silver-Layer String Cleaning Pattern

A better pattern than blindly applying functions is:

```text
Raw string
    ↓
Understand source format
    ↓
Handle NULL according to business rule
    ↓
Trim / normalise whitespace
    ↓
Parse structured components if required
    ↓
Standardise characters / case if required
    ↓
Reconstruct target value if required
    ↓
Validate
    ↓
Silver-ready value
```

---

# 15. Important Business Rules

Do not automatically:

```text
uppercase every string
remove every special character
remove every space
apply initcap() to every name
convert NULL to empty string
rewrite IDs without understanding their format
```

String cleaning must preserve business meaning.

For example:

```text
Phone formatting
→ may be removable

Country code
→ may be important

Address punctuation
→ may be meaningful

Product-code hyphen
→ may be part of the identifier
```

---

# 16. Quick Decision Guide

| Requirement | Function |
|---|---|
| Convert to uppercase | `upper()` |
| Convert to lowercase | `lower()` |
| Capitalise words | `initcap()` |
| Remove whitespace from both ends | `trim()` |
| Remove whitespace from left | `ltrim()` |
| Remove whitespace from right | `rtrim()` |
| Replace text using a regex | `regexp_replace()` |
| Break string into parts | `split()` |
| Safely retrieve split-array position | `F.get()` |
| Join string expressions | `concat()` |
| Join values with a separator | `concat_ws()` |

---

# 17. Easy Way to Remember

```text
Case problem?
→ upper() / lower() / initcap()

Outer whitespace?
→ trim() / ltrim() / rtrim()

Pattern cleaning?
→ regexp_replace()

Need parts of a string?
→ split()

Need a split result safely?
→ get()

Need to join values?
→ concat() / concat_ws()
```

---

# 18. Final S3.11 Understanding

```text
Raw Bronze String
        ↓
Understand format
        ↓
Trim / normalise
        ↓
Clean using regex when required
        ↓
Split structured values
        ↓
Safely access components
        ↓
Reconstruct when necessary
        ↓
Validate business meaning
        ↓
Silver-ready String
```

**S3.11 — String Functions and Silver-Layer String Cleaning completed.**